# 01. The protocol and the leakage test

How the backtest is built so that it cannot flatter the model: daily origins, features at
availability through a point in time frame, a leakage test that recomputes rows at their origin,
and the deliberately leaky lag that the frame refuses. The dead end is kept at the end: the first
threshold search, which was indifferent to the threshold because the weather columns were in the
wrong units.

Every number printed here is read from the committed manifest or the committed results; nothing
is refit.

In [ ]:
import json
from pathlib import Path

import numpy as np
import polars as pl

ROOT = Path.cwd() if (Path.cwd() / "results").exists() else Path.cwd().parent
manifest = json.loads((ROOT / "results" / "manifest.json").read_text(encoding="utf-8"))
values = manifest["values"]
v = lambda key: values[key]["value"]
print("as of", manifest["as_of"], "with", len(values), "values")

## Origins, horizons and windows

One origin per day per authority at the stated issue hour, horizons 1 to 48, the last twelve
months as the test period and the twelve before as validation.

In [ ]:
windows = json.loads((ROOT / "data" / "eia930" / "windows.json").read_text())
print({k: windows[k] for k in ("training_start", "validation_start", "test_start", "test_end")})
print("issue hour UTC", v("policy.issue_hour_utc"), "horizons", v("policy.horizons"))
print("origins in the own backtest", v("backtest.own.origins"), "fits", v("backtest.own.fits"))

## The frame refuses anything after the origin

`PointInTimeFrame` wraps a series and an origin; a read at or before the origin answers, a read
after it raises `LeakageError`. The same hour lag is 24 hours for the first day and 48 for the
second, because at a 00:00 origin the target hour of the second day has no yesterday yet.

In [ ]:
from datetime import UTC, datetime, timedelta

from lookahead_features.build import same_hour_lag
from lookahead_features.frame import LeakageError, PointInTimeFrame, SeriesIndex

hours = 24 * 40
start = datetime(2025, 1, 1, tzinfo=UTC)
demand = 10_000 + 2_000 * np.sin(2 * np.pi * np.arange(hours) / 24)
series = SeriesIndex(authority="DEMO", first_hour=start, values=demand.astype(float))
origin = start + timedelta(hours=24 * 30)
frame = PointInTimeFrame(series, origin)
print("horizon 1 same hour lag", same_hour_lag(1), "horizon 30 same hour lag", same_hour_lag(30))
print("lag 24 before a target 6 hours ahead:", frame.lag(origin + timedelta(hours=6), 24))
try:
    frame.lag(origin + timedelta(hours=6), 0)
except LeakageError as e:
    print("the leaky lag was refused:", e)

## The recomputation test

A sample of design rows is rebuilt from the raw series through the frame and compared with the
vectorized design; the registry's leakage gate runs the same function on a real authority.

In [ ]:
from lookahead_features.build import FeatureSpec
from lookahead_features.leakage import recomputation_check

rng = np.random.default_rng(0)
temperature = 10 + 12 * np.sin(2 * np.pi * np.arange(hours) / 24) + rng.normal(0, 1, hours)
humidity = np.full(hours, 60.0)
offsets = np.full(hours, -5)
origins = np.array([24 * d for d in range(20, 39)])
report = recomputation_check(series, temperature, humidity, offsets, FeatureSpec(), 10_000.0, origins, seed=1)
print("rows checked", report.rows_checked, "largest gap", report.max_abs_gap, "leaky lag refused", report.leaky_lag_refused, "green", report.green)

## The dead end: a threshold search that could not see the threshold

The first own model put raw degree hours beside a ratio target. On the simulator, where the true
heating threshold is known, the validation MAPE was flat across the candidate thresholds, so the
search picked whichever came first. Scaling the weather columns by the authority's typical demand
and replacing the raw temperature with the change in degree hours against the same hour lag made
the search recover the truth. The committed search grid shows the current spread across
thresholds for one real authority: the search now has something to choose.

In [ ]:
search = pl.read_parquet(ROOT / "results" / "backtest" / "own" / "threshold_search.parquet")
one = search.filter(pl.col("authority") == search["authority"][0])
best_penalty = one.sort("validation_mape")["penalty"][0]
grid = one.filter(pl.col("penalty") == best_penalty).pivot(on="cooling", index="heating", values="validation_mape")
print(one["authority"][0], "penalty", best_penalty)
print(grid)
spread = float(one["validation_mape"].max() - one["validation_mape"].min())
print("spread of validation MAPE across the grid:", round(spread, 4), "(a flat grid was the bug)")

The chosen thresholds per authority are in the manifest, and the protocol test holds every
choice to the validation year.

In [ ]:
print("chosen on validation:", v("backtest.own.chosen_on_validation"))
print("own MAPE", v("backtest.own.mape"), "seasonal naive", v("backtest.own.mape_naive"), "operator", v("backtest.own.mape_operator"))